# PreTest 03: Multiclass Classification — การจำแนกสายพันธุ์นกเพนกวิน
### Pipeline + ColumnTransformer · KNN / SVM / Gradient Boosting · GridSearchCV / RandomizedSearchCV · Metrics หลายคลาส

> **ที่มา:** EDA ดัดแปลงจาก Kaggle Notebook [penguin dataset : The new Iris](https://www.kaggle.com/code/parulpandey/penguin-dataset-the-new-iris) โดย Parul Pandey
> ส่วน Modeling, Evaluation และ Tuning เขียนเพิ่มทั้งหมด พร้อมคำอธิบายภาษาไทย
>
> **ชุดข้อมูล (Dataset):** [Palmer Archipelago (Antarctica) Penguin Data](https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data) — เพนกวิน 344 ตัว 3 สายพันธุ์

---

## ความแตกต่างจาก PreTest 01

| หัวข้อ | PreTest 01 (ฝน) | PreTest 03 (เพนกวิน) |
|---|---|---|
| จำนวนคลาส | 2 (Binary) | **3 (Multiclass)** |
| เตรียมข้อมูล | ทำทีละขั้นด้วยมือ | **Pipeline + ColumnTransformer** (อัตโนมัติ ไม่ Leak) |
| โมเดล | Logistic Regression, Decision Tree, Random Forest | **KNN, SVM, Gradient Boosting** |
| Tuning | GridSearchCV | GridSearchCV + **RandomizedSearchCV** + **Validation/Learning curve** |
| Metrics | Precision/Recall ของคลาสเดียว | **Macro vs Weighted average**, Confusion Matrix 3×3 |

## โครงบทเรียน
- **Part A (ขั้นที่ 1–10):** ใช้ Feature ครบทุกตัว → ทำครบทุกขั้นตอน
- **Part B (ขั้นที่ 11):** โจทย์ยากขึ้น (ตัด Feature ที่ช่วยมากออก) → ฝึกอ่านผล Multiclass จริงจัง

---
# ขั้นที่ 1: กำหนดโจทย์
- **เป้าหมาย:** ทำนาย **สายพันธุ์** (`species`) ของเพนกวินจากขนาดร่างกาย เกาะที่อยู่ และเพศ
- **ประเภทปัญหา:** Target มี 3 ค่า (Adelie, Chinstrap, Gentoo) → **Multiclass Classification**
- **วัดผลด้วย:** Accuracy, **Macro F1** (ให้ทุกสายพันธุ์สำคัญเท่ากัน), Confusion Matrix 3×3

# ขั้นที่ 2: นำเข้าไลบรารีและโหลดข้อมูล

---
# ขั้นที่ 3: สำรวจข้อมูลเบื้องต้น

| คอลัมน์ | ความหมาย | ชนิด |
|---|---|---|
| **`species`** | **สายพันธุ์ ← Target** (Adelie / Chinstrap / Gentoo) | Categorical |
| `island` | เกาะที่พบ (Biscoe / Dream / Torgersen) | Categorical |
| `culmen_length_mm` | ความยาวจะงอยปาก (มม.) | Numerical |
| `culmen_depth_mm` | ความหนาจะงอยปาก (มม.) | Numerical |
| `flipper_length_mm` | ความยาวครีบ/ปีก (มม.) | Numerical |
| `body_mass_g` | น้ำหนักตัว (กรัม) | Numerical |
| `sex` | เพศ (MALE / FEMALE) | Categorical |

> **Culmen** = สันบนของจะงอยปาก วัดได้ทั้งความยาว (length) และความหนา (depth)

> **ข้อสังเกต:** คอลัมน์ `sex` มีค่าผิดปกติ คือ **`'.'`** 1 แถว — นี่คือ **ข้อมูลสกปรก (Dirty data)** ที่ต้องแก้ในขั้นที่ 4

---
# ขั้นที่ 4: ทำความสะอาดข้อมูล

### 4.1 แถวที่ค่าวัดหายทั้งหมด

> **หมายเหตุ:** 2 แถวนี้ **ไม่มีค่าการวัดร่างกายแม้แต่ค่าเดียว** มีเพียงชื่อเกาะ → การเติมค่าเท่ากับการสร้างข้อมูลขึ้นเองเกือบทั้งแถว จึงลบออก

### 4.2 แก้ค่าที่ผิดรูปแบบ (`sex = '.'`)
ค่า `'.'` ไม่ใช่เพศที่ถูกต้อง → แปลงเป็นค่าหาย (`NaN`) แล้วให้ Pipeline เติมค่าในขั้นที่ 6

### 4.3 ข้อมูลซ้ำ

---
# ขั้นที่ 5: EDA & Visualization

### 5.1 จำนวนแต่ละสายพันธุ์ (Class balance)

> **ไม่สมดุลเล็กน้อย:** Chinstrap มีเพียง ~20% → ต้องใช้ `stratify` และดู **Macro F1**
> **`island` เป็นตัวบ่งชี้ที่สำคัญ:** Gentoo พบเพียงเกาะ Biscoe, Chinstrap พบเพียงเกาะ Dream, Torgersen มีแต่ Adelie

### 5.2 Pair Plot: ดูทุกคู่ตัวแปรพร้อมกัน

### 5.3 ตัวแปรที่แยกสายพันธุ์ได้ดี

> **Insight:**
> - **Gentoo** ครีบยาวและตัวหนักกว่าชัดเจน (แยกได้ง่ายที่สุด)
> - **Adelie vs Chinstrap** ขนาดตัวใกล้กันมาก แต่ **`culmen_length` (ปากยาว) ต่างกันชัด** — Chinstrap ปากยาวกว่า
> - ดังนั้น `culmen_length_mm` + `island` น่าจะเป็น Feature ที่สำคัญที่สุด

### 5.4 Correlation

> **ข้อสังเกต:** `flipper_length` กับ `body_mass` สัมพันธ์กันสูง (0.87) — ตัวใหญ่ครีบก็ยาว

---
# ขั้นที่ 6: เตรียมข้อมูลด้วย Pipeline + ColumnTransformer

### เหตุผลที่ใช้ Pipeline แทนการเตรียมข้อมูลทีละขั้นแบบ PreTest 01
| ทำทีละขั้นด้วยมือ | Pipeline |
|---|---|
| ต้องจำว่า fit กับ Train แล้ว transform Test ทุกขั้น | ทำให้อัตโนมัติ |
| ทำ Cross-Validation แล้ว **Leak** ง่าย (Scale ทั้ง Train ก่อน CV) | Preprocess ใหม่ **ในทุก Fold** → ไม่ Leak |
| ข้อมูลใหม่ต้องเตรียมเองอีกรอบ | เรียก `pipe.predict(ข้อมูลดิบ)` ได้ทันที |

```
ข้อมูลดิบ ─┬─ ตัวเลข  → [เติม median] → [StandardScaler] ─┐
           └─ ข้อความ → [เติม mode]   → [OneHotEncoder]  ─┴→ [Model] → คำทำนาย
```

### 6.1 แยก X, y และแบ่ง Train/Test

### 6.2 สร้าง Preprocessor

### 6.3 ฟังก์ชันวัดผล (ใช้ซ้ำกับทุกโมเดล)
- **Macro F1** = เฉลี่ย F1 ของ 3 สายพันธุ์ **แบบเท่ากัน** → Chinstrap (ตัวน้อย) สำคัญเท่า Adelie
- **Weighted F1** = เฉลี่ยโดย **ถ่วงน้ำหนักตามจำนวน** → คลาสที่มีมากมีผลมากกว่า

---
# ขั้นที่ 7: โมเดลที่ 1 — K-Nearest Neighbors (KNN)

### หลักการ
ทำนายจาก **"เพื่อนบ้านที่ใกล้ที่สุด k ตัว"** — ดูเพนกวินใน Train ที่ขนาดใกล้เคียงที่สุด k ตัว แล้วโหวตว่าส่วนใหญ่เป็นสายพันธุ์อะไร
- ใช้ **ระยะทาง** → **ต้อง Scale** (มิฉะนั้น `body_mass_g` ซึ่งมีค่าหลักพันจะมีอิทธิพลเหนือตัวแปรอื่น)
- `k` น้อย → ไวต่อ noise (Overfit), `k` มาก → เรียบเกินไป (Underfit)

### 7.1 Pipeline + Cross-Validation

### 7.2 Validation Curve: การเลือกค่า k
ใช้ `validation_curve` วาดคะแนน Train vs CV ที่ k ต่างๆ

> **หมายเหตุ:** ชื่อพารามิเตอร์ใน Pipeline ใช้รูปแบบ **`ชื่อขั้น__ชื่อพารามิเตอร์`** (ขีดล่าง 2 ตัว) เช่น `model__n_neighbors`

### 7.3 GridSearchCV หา k และวิธีวัดระยะ

### 7.4 การวิเคราะห์ผล KNN

| ประเด็น | ผลลัพธ์ | การตีความ |
|---|---|---|
| k ที่ดีที่สุด | `k = 7`, `weights='distance'` (CV Macro F1 **0.995**) | เพื่อนบ้านที่ใกล้กว่ามีสิทธิ์โหวตมากกว่า |
| Validation curve | k = 1–15 ได้ ~0.98–0.99, k ≥ 25 ลดลงเหลือ ~0.93 | k มากเกินไป → **Underfit** (เฉลี่ยจนเรียบเกินไป) |
| Test | Accuracy / Macro F1 = **1.000** (ถูกทั้ง 86 ตัว) | ทำนายถูกทั้งหมด |
| Train Acc = 1.000 แปลว่า Overfit หรือไม่ | ไม่ใช่ | เมื่อใช้ `weights='distance'` จุดใน Train ใกล้ตัวเองที่ระยะ 0 จึงได้ 100% เสมอ → **ดู CV และ Test แทน** |

> **สรุป:** KNN ทำได้สมบูรณ์แบบเพราะเพนกวินแต่ละสายพันธุ์อยู่เป็นกลุ่มชัดเจนในพื้นที่ Feature (ดูข้อ 5.2–5.3)

---
# ขั้นที่ 8: โมเดลที่ 2 — Support Vector Machine (SVM)

### หลักการ
หา **เส้นแบ่ง (Hyperplane) ที่ห่างจากทั้งสองฝั่งมากที่สุด** (Maximum margin)
- `kernel='linear'` = เส้นตรง / `kernel='rbf'` = เส้นโค้งได้
- `C` = ยอมให้ทำนายผิดได้เพียงใด (C มาก = เข้มงวด เสี่ยง Overfit)
- `gamma` (rbf) = ความโค้งของเส้น (gamma มาก = โค้งมาก เสี่ยง Overfit)
- ใช้ระยะทาง → **ต้อง Scale**, สำหรับหลายคลาส sklearn จะสร้างตัวแบ่งทีละคู่ให้อัตโนมัติ

### 8.1 RandomizedSearchCV: สุ่มทดลองค่า แทนการทดลองทุกค่า
| | GridSearchCV | RandomizedSearchCV |
|---|---|---|
| วิธี | ทดลอง **ทุก** ค่าในตาราง | **สุ่ม** ทดลอง `n_iter` ครั้งจากช่วงที่กำหนด |
| เหมาะกับ | พารามิเตอร์น้อย ค่าไม่มาก | พารามิเตอร์จำนวนมาก / ช่วงค่ากว้าง (เร็วกว่ามาก) |

### 8.2 การวิเคราะห์ผล SVM

| ประเด็น | ผลลัพธ์ | การตีความ |
|---|---|---|
| Hyperparameter ที่สุ่มเจอดีที่สุด | `kernel='linear'`, `C ≈ 85.7` (CV Macro F1 **0.995**) | เส้นแบ่งแบบเส้นตรงก็เพียงพอ → ข้อมูล **แยกด้วยเส้นตรงได้** (Linearly separable) |
| `gamma` | สุ่มได้ ~0.025 | ไม่มีผลเมื่อ kernel = linear (ใช้เฉพาะ rbf) |
| Test | Accuracy / Macro F1 = **1.000** | ทำนายถูกทั้งหมด |
| Overfit หรือไม่ | Train 1.000 = Test 1.000 | ไม่ |
| ใช้เวลาค้นหา | RandomizedSearchCV ทดลอง 30 ชุด (× 5 Fold) | เร็วกว่าทดลองทุกค่าแบบ Grid มาก |

> **สรุป:** SVM แบบเส้นตรงก็แยกได้ครบทั้ง 3 สายพันธุ์ — เป็นโมเดลที่เรียบง่ายและเหมาะที่สุดสำหรับข้อมูลชุดนี้

---
# ขั้นที่ 9: โมเดลที่ 3 — Gradient Boosting

### หลักการ
สร้าง Decision Tree ขนาดเล็ก **ทีละต้นต่อกัน** แต่ละต้นใหม่ **เรียนรู้จากความผิดพลาดของต้นก่อนหน้า**
(ต่างจาก Random Forest ที่สร้างหลายต้นพร้อมกันแล้วโหวต)
- `n_estimators` จำนวนต้น, `learning_rate` แต่ละต้นแก้ได้มากเพียงใด, `max_depth` ความลึกแต่ละต้น
- Tree ไม่ต้อง Scale แต่ใช้ Preprocessor เดิมได้ ไม่เสียหาย
- ตระกูลเดียวกับ **XGBoost / LightGBM** ที่ชนะการแข่งขันบ่อยๆ

### 9.1 Feature Importance (ดึงจาก Pipeline)

### 9.2 การวิเคราะห์ผล Gradient Boosting

| ประเด็น | ผลลัพธ์ | การตีความ |
|---|---|---|
| Hyperparameter ที่ดีที่สุด | 70 ต้น, `max_depth=2`, `learning_rate ≈ 0.12` | ต้นไม้ขนาดเล็กจำนวนไม่มากก็เพียงพอ |
| CV Macro F1 | **0.985** | ต่ำกว่า KNN/SVM (0.995) เล็กน้อย |
| Test | Accuracy / Macro F1 = **1.000** | ทำนายถูกทั้งหมด |
| Feature สำคัญ | `culmen_length` ~0.46 > `flipper_length` ~0.33 > `island_Dream` ~0.12 | ตรงกับ EDA: ปากยาวแยก Adelie/Chinstrap, ครีบยาวแยก Gentoo |
| Feature ที่แทบไม่ถูกใช้ | `body_mass_g`, `sex` ≈ 0 | `body_mass` ซ้ำซ้อนกับ `flipper_length` (corr 0.87) |

> **สรุป:** แม่นเท่ากันบน Test แต่ข้อดีพิเศษคือบอก **Feature Importance** ได้ — ใช้อธิบายว่าโมเดลแยกสายพันธุ์ด้วยอะไร

---
# ขั้นที่ 10: เปรียบเทียบ (Part A — Feature ครบ)

### 10.1 การวิเคราะห์ผล Part A

| โมเดล | CV Macro F1 | Test Acc | Test Macro F1 |
|---|---|---|---|
| KNN | 0.995 | 1.000 | 1.000 |
| SVM | 0.995 | 1.000 | 1.000 |
| Gradient Boosting | 0.985 | 1.000 | 1.000 |

**ข้อค้นพบ**
1. **ทั้ง 3 โมเดลทำนายถูกทั้ง 86 ตัว** — Confusion Matrix ไม่มีค่าใดอยู่นอกเส้นทแยงมุม
2. **คะแนน 100% ต้องตรวจสอบ Leakage ก่อนสรุปผล:** Feature ทุกตัวเป็นค่าที่วัดจากตัวเพนกวินจริง ไม่ได้สร้างจาก `species` → **ไม่ใช่ Leakage** แต่โจทย์ง่ายจริง (ดูกราฟ 5.3: ปากยาว + ครีบยาว แยกได้เกือบสมบูรณ์)
3. **ตัดสินจาก Test ไม่ได้** เพราะเสมอกันหมด → ดู **CV** แทน: KNN = SVM (0.995) > Gradient Boosting (0.985)
4. Test มีเพียง 86 ตัว — ทำนายผิด 1 ตัว Accuracy ลด ~1.2 จุด → ความแตกต่างเพียงเล็กน้อย อาจเป็นเพียงโชค

**เลือก:** **SVM (linear)** — CV สูงสุดเท่ากับ KNN แต่เป็นเส้นแบ่งตรงที่เรียบง่ายกว่า และทำนายเร็วกว่า KNN เมื่อข้อมูลมาก
→ แต่โจทย์ง่ายเกินไปที่จะเห็นความต่างของโมเดล ไปดู **Part B** ที่ยากขึ้น

---
# ขั้นที่ 11 (Part B): โจทย์ที่ซับซ้อนขึ้น — กรณีวัดได้เฉพาะขนาดร่างกาย

**สถานการณ์:** นักวิจัยวัดได้เพียง `culmen_depth`, `flipper_length`, `body_mass` (ไม่มีความยาวปาก ไม่รู้ว่ามาจากเกาะไหน)
→ Adelie กับ Chinstrap ขนาดตัวใกล้กันมาก (ขั้นที่ 5.3) โมเดลจะสับสนหรือไม่?

### 11.1 เทรน 3 โมเดลใหม่ด้วย Feature ชุดเล็ก

### 11.2 Confusion Matrix 3×3: คู่สายพันธุ์ที่โมเดลสับสน
อ่านแบบเดิม: **แถว = ค่าจริง, คอลัมน์ = ค่าที่ทำนาย** ตัวเลขนอกเส้นทแยง = ทำนายผิด

### 11.3 Classification Report ของโมเดลที่ดีที่สุด (Macro vs Weighted)

### 11.4 Learning Curve: ผลของการเพิ่มปริมาณข้อมูล
- ถ้าเส้น Train กับ CV **ยังห่างกันมาก** และ CV ยังขึ้นอยู่ → เพิ่มข้อมูลน่าจะช่วย
- ถ้าเส้น **ชนกันแล้วแบนราบ** → เพิ่มข้อมูลไม่ช่วย ต้องหา Feature ใหม่หรือเปลี่ยนโมเดล

### 11.5 การวิเคราะห์ผล Part B

| โมเดล | CV Macro F1 | Train Acc | Test Acc | Macro F1 | Weighted F1 | Recall Chinstrap |
|---|---|---|---|---|---|---|
| KNN | 0.788 | 0.852 | **0.814** | **0.756** | **0.806** | **0.41** (7/17) |
| SVM | 0.778 | 0.852 | 0.779 | 0.672 | 0.751 | 0.18 (3/17) |
| Gradient Boosting | **0.789** | 0.945 | 0.767 | 0.681 | 0.751 | 0.24 (4/17) |

**อ่าน Confusion Matrix 3×3**
- **Gentoo ถูกหมดทุกโมเดล (31/31)** — ตัวใหญ่ ครีบยาว แยกออกง่ายแม้ไม่มีความยาวปาก
- **ความผิดพลาดเกือบทั้งหมดคือ Adelie ↔ Chinstrap** — ขนาดตัวใกล้กันมาก (ข้อ 5.3) โดยเฉพาะ **Chinstrap ถูกทำนายเป็น Adelie** 10–14 จาก 17 ตัว

**Macro vs Weighted F1**
- Weighted F1 (0.75–0.81) **สูงกว่า** Macro F1 (0.67–0.76) เพราะคลาสที่ทำนายได้ต่ำที่สุด (Chinstrap) มีจำนวนน้อยที่สุด → ถูกถ่วงน้ำหนักน้อย
- ถ้ารายงานเพียง Accuracy (~0.78–0.81) หรือ Weighted F1 จะ **มองไม่เห็นว่า Chinstrap แทบทำนายไม่ได้** → ใช้ **Macro F1 + Recall รายคลาส**

**การเลือกโมเดล**
- CV เลือก Gradient Boosting (0.789) แต่ KNN (0.788) **ต่างกันเพียง 0.001 = ถือว่าเสมอกัน**
- Gradient Boosting **Overfit** (Train 0.945 vs Test 0.767) ขณะที่ KNN ช่องว่างน้อยกว่า (0.852 vs 0.814) และทำนาย Chinstrap ได้ดีที่สุด
- → เมื่อคะแนน CV เสมอกัน ให้เลือกตัวที่ **Overfit น้อยกว่า / เรียบง่ายกว่า** = **KNN**

**Learning curve (Gradient Boosting)**
- CV เพิ่มจาก ~0.71 → ~0.78 เมื่อข้อมูลเพิ่ม และยังห่างจาก Train (~0.95) มาก → **เพิ่มข้อมูลน่าจะช่วยได้อีก**

> **บทเรียนสำคัญ:** เพิ่ม `island` กลับเข้าไป (แบบฝึกหัดข้อ 1) Test Acc ขึ้นเป็น **0.907** และ Macro F1 **0.882**
> → **การเลือก Feature ที่ดีสำคัญกว่าการเลือกโมเดล** (เปลี่ยนโมเดลได้ต่างกันเพียง ~5 จุด แต่เพิ่ม Feature เดียวได้เพิ่ม ~10 จุดขึ้นไป)

---
# ขั้นที่ 12: ใช้งานโมเดล — ทำนายเพนกวินตัวใหม่ และบันทึกโมเดล
Pipeline รับ **ข้อมูลดิบ** ได้โดยตรง (แม้มีค่าหาย) ไม่ต้องเตรียมข้อมูลเอง

---
## รายการตรวจสอบ (Checklist): Multiclass Classification
- [ ] ตรวจค่าแปลกปลอม (เช่น `'.'`) ไม่ใช่เพียง `NaN`
- [ ] `stratify=y` ตอนแบ่งข้อมูล
- [ ] ใช้ **Pipeline + ColumnTransformer** → CV ไม่ Leak และทำนายข้อมูลดิบได้
- [ ] KNN / SVM ต้อง Scale, Tree ไม่ต้อง
- [ ] Tuning: GridSearchCV (ค่าน้อย) / RandomizedSearchCV (ค่าจำนวนมาก) ชื่อพารามิเตอร์ `model__xxx`
- [ ] รายงาน **Macro F1** + Confusion Matrix 3×3 + Classification Report
- [ ] ถ้าคะแนน 100% ให้ถามว่า "โจทย์ง่ายจริง หรือมี Leakage?"

## แบบฝึกหัด
1. ใน Part B ทดลองเพิ่ม `island` กลับเข้าไป (`make_preprocessor(num_B, ['island'])`) — คะแนนดีขึ้นเท่าใด
2. ทดลองเปลี่ยนโมเดลเป็น `RandomForestClassifier` และ `LogisticRegression` ใน Pipeline เดียวกัน